# 02 — Chunking y generación de embeddings

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Prototipado del pipeline de embeddings

---

Este notebook cubre la segunda fase del pipeline RAG: la división del documento
extraído en chunks y la generación de embeddings semánticos mediante **BGE-M3** (BAAI).

**Entrada:** `data/processed/PMC10967698_extracted.md` (generado en `01_ingesta.ipynb`)  
**Salida:** vectores de embeddings listos para indexar en ChromaDB (`03_recuperacion.ipynb`)

In [ ]:
"""
Notebook: 02_embeddings.ipynb

Objetivo:
    Dividir el documento Markdown extraído en chunks de tamaño óptimo
    y generar embeddings semánticos usando el modelo BGE-M3 (BAAI),
    tal y como se haría en la fase de indexación de un pipeline RAG
    en producción.

    Este notebook cubre:
      1. Carga del documento procesado
      2. Chunking con solapamiento controlado
      3. Carga del modelo BGE-M3
      4. Generación de embeddings por chunk
      5. Validación dimensional y semántica
      6. Persistencia de embeddings para el siguiente paso

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-04-30
Versión: 1.0.0
"""

## 1. Configuración del entorno

In [1]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [2]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado correctamente


In [3]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

# Detección de GPU con PyTorch — relevante en embeddings porque
# acelera significativamente la generación de vectores.
# Si no hay GPU disponible, el pipeline continúa en CPU sin cambios.
import torch

if torch.cuda.is_available():
    dispositivo = "cuda"
    nombre_gpu = torch.cuda.get_device_name(0)
    print(f"Dispositivo : GPU — {nombre_gpu}")
else:
    dispositivo = "cpu"
    print("Dispositivo : CPU (correcto para prototipado)")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.113+
Dispositivo : CPU (correcto para prototipado)


## 2. Instalación de dependencias

In [4]:
# FlagEmbedding: librería oficial de BAAI para BGE-M3
# sentence-transformers: interfaz estándar para modelos de embeddings
%pip install FlagEmbedding sentence-transformers -q

## 3. Definición de rutas y parámetros

In [5]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Ruta al documento procesado (output de 01_ingesta.ipynb)
DIR_PROCESSED = PROYECTO_RAIZ / 'data' / 'processed'
RUTA_MD = DIR_PROCESSED / 'PMC10967698_extracted.md'

# Ruta de salida para los embeddings generados
DIR_EMBEDDINGS = PROYECTO_RAIZ / 'data' / 'embeddings'
DIR_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

# Parámetros de chunking
# chunk_size: número máximo de caracteres por fragmento
# chunk_overlap: solapamiento entre chunks consecutivos para preservar contexto
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 100

# Modelo de embeddings
MODELO_EMBEDDINGS = 'BAAI/bge-m3'

# Validación de existencia del archivo de entrada
assert RUTA_MD.exists(), (
    f"Archivo no encontrado: {RUTA_MD}\n"
    f"Ejecuta primero el notebook 01_ingesta.ipynb"
)

print(f"Documento de entrada  : {RUTA_MD.name}")
print(f"Directorio de salida  : {DIR_EMBEDDINGS}")
print(f"Chunk size            : {CHUNK_SIZE} caracteres")
print(f"Chunk overlap         : {CHUNK_OVERLAP} caracteres")
print(f"Modelo de embeddings  : {MODELO_EMBEDDINGS}")

Documento de entrada  : PMC10967698_extracted.md
Directorio de salida  : /content/drive/MyDrive/chem-rag-assistant/data/embeddings
Chunk size            : 1000 caracteres
Chunk overlap         : 100 caracteres
Modelo de embeddings  : BAAI/bge-m3


## 4. Carga del documento procesado

In [6]:
# Lectura del documento Markdown extraído en el notebook anterior
with open(RUTA_MD, 'r', encoding='utf-8') as f:
    documento_md = f.read()

print(f"Documento cargado correctamente")
print(f"Caracteres totales    : {len(documento_md):,}")
print(f"Palabras aproximadas  : {len(documento_md.split()):,}")

Documento cargado correctamente
Caracteres totales    : 49,922
Palabras aproximadas  : 8,883


## 5. Chunking del documento

In [7]:
# Instalación de LangChain y su módulo de text splitting
# Versiones recientes de LangChain: el splitter está en langchain-text-splitters
%pip install langchain langchain-text-splitters -q

# Import del splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("RecursiveCharacterTextSplitter importado correctamente")

RecursiveCharacterTextSplitter importado correctamente


In [15]:
# Inicialización del splitter con los parámetros definidos
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "],
)

# Generación de chunks
chunks = splitter.split_text(documento_md)

print(f"Total de chunks generados : {len(chunks)}")
print(f"Tamaño medio por chunk    : {sum(len(c) for c in chunks) // len(chunks)} caracteres")
print(f"Chunk más corto           : {min(len(c) for c in chunks)} caracteres")
print(f"Chunk más largo           : {max(len(c) for c in chunks)} caracteres")

Total de chunks generados : 87
Tamaño medio por chunk    : 600 caracteres
Chunk más corto           : 15 caracteres
Chunk más largo           : 999 caracteres


In [16]:
# Fusión de chunks cortos con el chunk siguiente
# Los chunks cortos son títulos de sección (## ...) que el splitter
# separó del contenido. Se fusionan con el chunk siguiente para
# preservar el contexto semántico del encabezado junto a su contenido,
# ya que un título aislado no es útil para la recuperación semántica.
LONGITUD_MINIMA = 100

chunks_fusionados = []
i = 0
while i < len(chunks):
    if len(chunks[i]) < LONGITUD_MINIMA and i + 1 < len(chunks):
        # Fusionar título con el chunk siguiente
        chunks_fusionados.append(chunks[i] + "\n" + chunks[i + 1])
        i += 2
    else:
        chunks_fusionados.append(chunks[i])
        i += 1

print(f"Chunks originales         : {len(chunks)}")
print(f"Chunks fusionados         : {len(chunks) - len(chunks_fusionados)}")
print(f"Chunks resultantes        : {len(chunks_fusionados)}")
print(
    f"Tamaño medio post-fusión  : "
    f"{sum(len(c) for c in chunks_fusionados) // len(chunks_fusionados)} caracteres"
)

Chunks originales         : 87
Chunks fusionados         : 10
Chunks resultantes        : 77
Tamaño medio post-fusión  : 678 caracteres


In [14]:
# Inspección del primer chunk para verificar la calidad del splitting
print("=" * 60)
print("PREVIEW — CHUNK 1")
print("=" * 60)
print(chunks_fusionados[1])
print("=" * 60)

PREVIEW — CHUNK 1
‡ These authors contributed equally to this work.

Synthesis, characterization, and biomedical evaluation of ethylene-bridged tetra-NHC Pd(II), Pt(II) and Au(III) complexes, with apoptosis-inducing properties in cisplatin-resistant neuroblastoma cells †

Wolfgang R. E. Büchele, ‡ a Tim P. Schlachta, ‡ a Andreas L. Gebendorfer, a Jenny Pamperin, cd Leon F. Richter, a Michael J. Sauer, a Aram Prokop* bcd and Fritz E. Kühn * a


## 6. Carga del modelo BGE-M3

In [17]:
# Carga del modelo BGE-M3 desde HuggingFace
# BGE-M3 es multilingüe, soporta hasta 8192 tokens por chunk
# y lidera los benchmarks de embeddings open-source en 2024-2025
from FlagEmbedding import BGEM3FlagModel

print(f"Cargando modelo {MODELO_EMBEDDINGS}...")
print("(La primera ejecución descarga ~2.2 GB — puede tardar varios minutos)")

modelo = BGEM3FlagModel(
    MODELO_EMBEDDINGS,
    use_fp16=True,      # Reduce uso de memoria a la mitad sin pérdida significativa
    device=dispositivo
)

print(f"Modelo cargado correctamente en: {dispositivo.upper()}")

Cargando modelo BAAI/bge-m3...
(La primera ejecución descarga ~2.2 GB — puede tardar varios minutos)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo cargado correctamente en: CPU


## 7. Generación de embeddings

In [18]:
# Generación de embeddings para todos los chunks fusionados
# Se usa chunks_fusionados (no chunks) para garantizar que cada
# vector representa un fragmento semánticamente completo,
# con títulos de sección fusionados a su contenido correspondiente.
# batch_size=4 es conservador para CPU — aumentar a 8-16 si hay GPU
print("Generando embeddings...")

resultado_embeddings = modelo.encode(
    chunks_fusionados,
    batch_size=4,
    max_length=512,      # Suficiente para nuestros chunks (~150 tokens de media)
    return_dense=True,   # Vectores densos para ChromaDB
    return_sparse=False, # No necesario para este pipeline
    return_colbert_vecs=False,
)

# Los embeddings densos son los vectores que indexaremos en ChromaDB
embeddings = resultado_embeddings['dense_vecs']

print(f"Embeddings generados correctamente")
print(f"Forma del array       : {embeddings.shape}")
print(f"Chunks procesados     : {embeddings.shape[0]}")
print(f"Dimensión del vector  : {embeddings.shape[1]}")

Generando embeddings...


Inference Embeddings: 100%|██████████| 20/20 [03:24<00:00, 10.22s/it]

Embeddings generados correctamente
Forma del array       : (77, 1024)
Chunks procesados     : 77
Dimensión del vector  : 1024


## 8. Validación de embeddings

In [19]:
# Validación dimensional: cada chunk debe tener un vector de 1024 dimensiones
# que es la dimensión estándar de BGE-M3
import numpy as np
from numpy.linalg import norm

DIMENSION_ESPERADA = 1024

assert embeddings.shape[0] == len(chunks_fusionados), (
    f"Error: número de embeddings ({embeddings.shape[0]}) "
    f"no coincide con número de chunks ({len(chunks_fusionados)})"
)
assert embeddings.shape[1] == DIMENSION_ESPERADA, (
    f"Error: dimensión inesperada ({embeddings.shape[1]}), "
    f"se esperaban {DIMENSION_ESPERADA}"
)

print("Validación dimensional : OK")


def similitud_coseno(v1, v2):
    """Calcula la similitud coseno entre dos vectores."""
    return float(np.dot(v1, v2) / (norm(v1) * norm(v2)))


# Prueba semántica: verifica que los vectores son válidos
# (no son ceros ni copias idénticas entre chunks distintos)
# La evaluación semántica comparativa se realizará en 02b_chunking_eval.ipynb
sim_01 = similitud_coseno(embeddings[0], embeddings[1])
sim_0_ultimo = similitud_coseno(embeddings[0], embeddings[-1])

assert 0.0 < sim_01 < 1.0, "Error: similitud chunk 0-1 fuera de rango esperado"
assert 0.0 < sim_0_ultimo < 1.0, "Error: similitud chunk 0-final fuera de rango esperado"

print(f"Similitud chunk 0 - chunk 1    : {sim_01:.4f}")
print(f"Similitud chunk 0 - chunk final: {sim_0_ultimo:.4f}")
print("Validación semántica   : OK")

Validación dimensional : OK
Similitud chunk 0 - chunk 1    : 0.5433
Similitud chunk 0 - chunk final: 0.5699
Validación semántica   : OK


## 9. Persistencia de embeddings y chunks

In [20]:
# Guardado de embeddings y chunks en formato numpy y JSON
# Estos archivos serán la entrada del notebook 03_recuperacion.ipynb
import json

# Embeddings en formato numpy (.npy)
RUTA_EMBEDDINGS = DIR_EMBEDDINGS / 'PMC10967698_embeddings.npy'
np.save(str(RUTA_EMBEDDINGS), embeddings)

# Chunks en formato JSON para mantener la correspondencia índice-texto
RUTA_CHUNKS = DIR_EMBEDDINGS / 'PMC10967698_chunks.json'
with open(RUTA_CHUNKS, 'w', encoding='utf-8') as f:
    json.dump(chunks_fusionados, f, ensure_ascii=False, indent=2)

print(f"Embeddings guardados en : {RUTA_EMBEDDINGS.name}")
print(f"Chunks guardados en     : {RUTA_CHUNKS.name}")
print(f"Tamaño embeddings       : {RUTA_EMBEDDINGS.stat().st_size / 1024:.1f} KB")
print(f"Tamaño chunks           : {RUTA_CHUNKS.stat().st_size / 1024:.1f} KB")

Embeddings guardados en : PMC10967698_embeddings.npy
Chunks guardados en     : PMC10967698_chunks.json
Tamaño embeddings       : 308.1 KB
Tamaño chunks           : 51.7 KB


## 10. Resumen de la ejecución

In [21]:
# Resumen final del proceso de embeddings
print("=" * 60)
print("RESUMEN — EMBEDDINGS COMPLETADOS")
print("=" * 60)
print(f"  Documento origen      : PMC10967698_extracted.md")
print(f"  Chunks generados      : {len(chunks_fusionados)}")
print(f"  Modelo utilizado      : {MODELO_EMBEDDINGS}")
print(f"  Dimensión del vector  : {embeddings.shape[1]}")
print(f"  Dispositivo           : {dispositivo.upper()}")
print(f"  Outputs generados     : embeddings.npy + chunks.json")
print("=" * 60)
print("Siguiente paso: 03_recuperacion.ipynb")

RESUMEN — EMBEDDINGS COMPLETADOS
  Documento origen      : PMC10967698_extracted.md
  Chunks generados      : 77
  Modelo utilizado      : BAAI/bge-m3
  Dimensión del vector  : 1024
  Dispositivo           : CPU
  Outputs generados     : embeddings.npy + chunks.json
Siguiente paso: 03_recuperacion.ipynb
